In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================================
# ICTSP ZERO-SHOT LEARNING - ETTh1 ↔ ETTm1 ONLY
# 2 Directions × 2 Pred Lengths × 4 Methods = 16 Experiments
# Methods: Baseline, STL, Wavelet, SSA
# ============================================================================

import os
import gc
import json
import time
import random
import warnings
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.interpolate import interp1d

warnings.filterwarnings("ignore")

# ============================================================================
# DEVICE SETUP
# ============================================================================
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = "cpu"
print(f"Using device: {device} (CPU for stability)")

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(2024)

# ============================================================================
# OPTIONAL PACKAGES
# ============================================================================
try:
    import pywt
    HAS_PYWT = True
except ImportError:
    HAS_PYWT = False
    print("PyWavelets not found. Installing...")
    subprocess.check_call(["pip", "install", "-q", "PyWavelets"])
    import pywt
    HAS_PYWT = True

try:
    from statsmodels.tsa.seasonal import STL as STLModel
    HAS_STL = True
except ImportError:
    HAS_STL = False
    print("statsmodels not found. Installing...")
    subprocess.check_call(["pip", "install", "-q", "statsmodels"])
    from statsmodels.tsa.seasonal import STL as STLModel
    HAS_STL = True


# ============================================================================
# RECONSTRUCTION METHODS
# ============================================================================

class STLDecomposition:
    @staticmethod
    def get_period(dataset_name: str) -> int:
        if "ETTh" in dataset_name:
            return 24  # Hourly data: 24 hours/day
        elif "ETTm" in dataset_name:
            return 96  # 15-min data: 96 periods/day
        return 24

    @staticmethod
    def denoise_signal(signal: np.ndarray, period: int, robust: bool = True) -> np.ndarray:
        if not HAS_STL:
            return signal
        signal = signal.copy()
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8 or len(signal) < 2 * period:
            return signal.astype(np.float32)
        try:
            stl = STLModel(signal, period=period, robust=robust)
            result = stl.fit()
            denoised = result.trend + result.seasonal
            bad = np.isnan(denoised)
            denoised[bad] = signal[bad]
            return denoised.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, dataset_name: str, robust: bool = True) -> np.ndarray:
        period = STLDecomposition.get_period(dataset_name)
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = STLDecomposition.denoise_signal(data[:, c], period, robust)
        return out


class WaveletReconstruction:
    @staticmethod
    def denoise_signal(signal: np.ndarray, wavelet: str = "db4", level: int = 2, threshold_factor: float = 0.1) -> np.ndarray:
        if not HAS_PYWT:
            return signal
        signal = signal.copy()
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8:
            return signal.astype(np.float32)
        max_level = max(1, int(np.log2(len(signal))) - 1)
        level = min(level, max_level)
        try:
            coeffs = pywt.wavedec(signal, wavelet, level=level)
            if len(coeffs) > 1:
                sigma = np.median(np.abs(coeffs[-1])) / 0.6745
                threshold = sigma * np.sqrt(2 * np.log(len(signal))) * threshold_factor
                coeffs_thresh = [coeffs[0]]
                for i in range(1, len(coeffs)):
                    coeffs_thresh.append(pywt.threshold(coeffs[i], threshold, mode="soft"))
                out = pywt.waverec(coeffs_thresh, wavelet)[:len(signal)]
            else:
                out = signal
            return out.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, wavelet: str = "db4", level: int = 2, threshold_factor: float = 0.1) -> np.ndarray:
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = WaveletReconstruction.denoise_signal(data[:, c], wavelet, level, threshold_factor)
        return out


class SSARecovery:
    @staticmethod
    def _embed(signal: np.ndarray, L: int) -> np.ndarray:
        N = len(signal)
        K = N - L + 1
        X = np.zeros((L, K), dtype=np.float32)
        for i in range(L):
            X[i, :] = signal[i:i + K]
        return X

    @staticmethod
    def _diagonal_averaging(X: np.ndarray) -> np.ndarray:
        L, K = X.shape
        N = L + K - 1
        signal = np.zeros(N, dtype=np.float32)
        counts = np.zeros(N, dtype=np.float32)
        for i in range(L):
            for j in range(K):
                signal[i + j] += X[i, j]
                counts[i + j] += 1.0
        return signal / np.maximum(counts, 1e-8)

    @staticmethod
    def denoise_signal(signal: np.ndarray, L: int = 30, r: int = 5) -> np.ndarray:
        signal = signal.copy()
        N = len(signal)
        if N < L:
            L = max(2, N // 2)
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8:
            return signal.astype(np.float32)
        try:
            X = SSARecovery._embed(signal, L)
            U, S, Vt = np.linalg.svd(X, full_matrices=False)
            r = min(r, len(S))
            Xr = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
            out = SSARecovery._diagonal_averaging(Xr)[:N]
            return out.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, L: int = 30, r: int = 5) -> np.ndarray:
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = SSARecovery.denoise_signal(data[:, c], L, r)
        return out


# ============================================================================
# DATA LOADING
# ============================================================================

class DataManager:
    DATASET_URLS = {
        "ETTh1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv",
        "ETTm1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv",
    }
    
    @staticmethod
    def load_raw(name: str) -> np.ndarray:
        url = DataManager.DATASET_URLS[name]
        print(f"Loading {name} from {url}...")
        df = pd.read_csv(url)
        if "date" in df.columns:
            df = df.drop(columns=["date"])
        values = df.values.astype(np.float32)
        print(f"✅ Loaded {name}: {values.shape[1]} channels, {values.shape[0]} time steps")
        return values


class WindowDataset(Dataset):
    def __init__(self, data: np.ndarray, seq_len: int, pred_len: int):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.window_len = seq_len + pred_len
        
        if len(data) >= self.window_len:
            self.n_windows = len(data) - self.window_len + 1
        else:
            self.n_windows = 0
    
    def __len__(self):
        return self.n_windows
    
    def __getitem__(self, idx):
        window = self.data[idx:idx + self.window_len]
        x_enc = window[:self.seq_len]
        y = window[self.seq_len:self.seq_len + self.pred_len]
        x_mark_dec_dummy = torch.zeros(self.seq_len + self.pred_len, 1)
        return x_enc, y, x_mark_dec_dummy


# ============================================================================
# ICTSP CORE
# ============================================================================

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_size, heads, dropout=0.1):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        self.dropout = nn.Dropout(dropout)
        assert self.head_dim * heads == embed_size
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)
    
    def forward(self, values, keys, queries, mask=None):
        N = queries.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], queries.shape[1]
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = queries.reshape(N, query_len, self.heads, self.head_dim)
        values = self.values(values)
        keys = self.dropout(self.keys(keys))
        queries = self.queries(queries)
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))
        energy = energy / (self.embed_size ** 0.5)
        attention = F.softmax(energy, dim=-1)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )
        out = self.fc_out(out)
        return out, attention


class TransformerEncoder(nn.Module):
    def __init__(self, emb_size=128, depth=3, heads=8, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=emb_size,
                nhead=heads,
                dim_feedforward=mlp_ratio * emb_size,
                batch_first=True,
                dropout=dropout,
                norm_first=False,
                activation="gelu",
            )
            for _ in range(depth)
        ])
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class Tokenizer(nn.Module):
    def __init__(self, lookback=512, output=96, stride=8):
        super().__init__()
        self.d = lookback + output
        self.s = stride
    
    def forward(self, tensor):
        B, C, L = tensor.shape
        if L >= self.d:
            unfolded = tensor.flip(-1).unfold(dimension=2, size=self.d, step=self.s)
            return unfolded.flip(-1).flip(-2)
        else:
            padding = torch.zeros(B, C, self.d - L, device=tensor.device, dtype=tensor.dtype)
            tensor_padded = torch.cat([tensor, padding], dim=-1)
            unfolded = tensor_padded.flip(-1).unfold(dimension=2, size=self.d, step=self.s)
            return unfolded.flip(-1).flip(-2)


class ICTSP(nn.Module):
    def __init__(
        self,
        lookback=512,
        output=96,
        depth=3,
        heads=8,
        mlp_ratio=4,
        d_model=128,
        external_stride=8,
        n_channels=7,
        dropout=0.5,
    ):
        super().__init__()
        self.lookback = lookback
        self.pred_len = output
        self.external_stride = external_stride
        self.input_projection = nn.Linear(lookback + output, d_model)
        self.transformer_encoder = TransformerEncoder(d_model, depth, heads, mlp_ratio, dropout)
        self.input_norm = nn.LayerNorm(d_model)
        self.output_norm = nn.LayerNorm(d_model)
        self.output_embedding = nn.Parameter(0.01 * torch.randn(1, 1, 1200))
        self.output_projection = nn.Linear(d_model, output)
        self.channel_embedding = nn.Parameter(0.01 * torch.randn(1, 1024, d_model))
        self.pos_embedding = nn.Parameter(0.01 * torch.randn(1, 8192, 1, d_model))
        self.pos_embedding_after = nn.Parameter(0.01 * torch.randn(1, 8192, d_model))
        self.linear_warmup_steps = 5000
        self.linear_warm_up_counter = 0
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
    
    def forward(self, x, x_mark_dec=None):
        lookback = self.lookback
        future = self.pred_len
        mean = x[:, [-1], :].detach()
        x = x.permute(0, 2, 1)
        output_embedding = self.output_embedding[:, :, 0:future].expand(x.shape[0], x.shape[1], -1)
        x = torch.cat([x, output_embedding + mean.permute(0, 2, 1)], dim=-1)
        B, C, _ = x.shape
        number_of_targets = C
        x_orig = x[:, :, 0:-future].clone()
        external_tokenizer = Tokenizer(lookback, future, stride=self.external_stride)
        ex_tokens = external_tokenizer(x_orig)
        _, _, _, d = ex_tokens.shape
        ex_tokens = ex_tokens.permute(0, 2, 1, 3).reshape(B, -1, d)
        x_target = x[:, -number_of_targets:, -(lookback + future):]
        x_tokens = torch.cat([ex_tokens, x_target], dim=1)
        token_mean = x_tokens[:, :, [-(future + 1)]].detach()
        x_tokens = x_tokens - token_mean
        x_tokens = self.input_projection(x_tokens)
        channel_mask = self.channel_embedding[:, -C:, :]
        x_tokens = x_tokens + channel_mask.repeat(1, x_tokens.shape[1] // C, 1)
        pos_emb = self.pos_embedding[:, -(x_tokens.shape[1] // C):, :, :].expand(-1, -1, C, -1)
        pos_emb = pos_emb.reshape(pos_emb.shape[0], pos_emb.shape[1] * pos_emb.shape[2], pos_emb.shape[3])
        x_tokens = x_tokens + pos_emb
        
        if self.linear_warm_up_counter < self.linear_warmup_steps:
            if self.training:
                self.linear_warm_up_counter += 1
            x_output = self.output_projection(x_tokens[:, -number_of_targets:, :])
            x_output = x_output + token_mean[:, -x_output.shape[1]:, :]
            x_output = x_output.permute(0, 2, 1)
            return x_output
        
        x_tokens = self.input_norm(x_tokens)
        x_tokens = self.transformer_encoder(x_tokens)
        x_output = x_tokens[:, -number_of_targets:, :]
        x_output = self.output_norm(x_output)
        x_output = self.output_projection(x_output)
        x_output = x_output + token_mean[:, -x_output.shape[1]:, :]
        x_output = x_output.permute(0, 2, 1)
        return x_output


class RepoICTSPModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = ICTSP(
            lookback=cfg.lookback,
            output=cfg.pred_len,
            depth=cfg.e_layers,
            heads=cfg.n_heads,
            mlp_ratio=cfg.mlp_ratio,
            d_model=cfg.d_model,
            external_stride=cfg.sampling_step,
            n_channels=cfg.enc_in,
            dropout=cfg.dropout,
        )
    
    def forward(self, x, x_mark_dec):
        return self.model(x, x_mark_dec)


# ============================================================================
# BASELINE MODELS (for comparison)
# ============================================================================

class LastValueBaseline(nn.Module):
    def forward(self, history, horizon):
        return history[:, -1:, :].repeat(1, horizon, 1)


class NLinearBaseline(nn.Module):
    def __init__(self, seq_len, horizon, num_channels):
        super().__init__()
        self.linears = nn.ModuleList([nn.Linear(seq_len, horizon) for _ in range(num_channels)])
    
    def forward(self, history):
        return torch.stack([self.linears[c](history[:, :, c]) for c in range(history.shape[-1])], dim=2)


# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    dataset_name: str = "ETTh1"
    experiment_mode: str = "zero_shot"
    use_stl: bool = False
    use_wavelet: bool = False
    use_ssa: bool = False
    lookback: int = 512
    pred_len: int = 96
    e_layers: int = 3
    d_model: int = 128
    n_heads: int = 8
    mlp_ratio: int = 4
    dropout: float = 0.5
    sampling_step: int = 8
    batch_size: int = 32
    batch_size_test: int = 32
    learning_rate: float = 5e-4
    weight_decay: float = 1e-5
    train_epochs: int = 100
    patience_steps: int = 20
    grad_clip: float = 1.0
    enc_in: int = 7
    device: str = device
    seed: int = 2024

    
    def __post_init__(self):
        self.lookback = 512
        self.batch_size = 32
        self.train_epochs = 100
        self.patience_steps = 20


# ============================================================================
# METRICS & EVALUATION
# ============================================================================

def compute_metrics(pred, target):
    with torch.no_grad():
        mse = F.mse_loss(pred, target).item()
        mae = F.l1_loss(pred, target).item()
    return mse, mae


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_mse = 0.0
    total_mae = 0.0
    n_batches = 0
    
    for x_enc, y, x_mark_dec in loader:
        x_enc = x_enc.to(device)
        y = y.to(device)
        x_mark_dec = x_mark_dec.to(device)
        pred = model(x_enc, x_mark_dec)
        mse, mae = compute_metrics(pred, y)
        total_mse += mse
        total_mae += mae
        n_batches += 1
    
    return {
        "mse": total_mse / n_batches if n_batches > 0 else float("inf"),
        "mae": total_mae / n_batches if n_batches > 0 else float("inf"),
    }


# ============================================================================
# ZERO-SHOT TRAINER
# ============================================================================

class ZeroShotTrainer:
    def __init__(self, cfg: Config, source_name: str, target_name: str, method_name: str):
        self.cfg = cfg
        self.device = cfg.device
        self.source_name = source_name
        self.target_name = target_name
        self.method_name = method_name
        set_seed(cfg.seed)
        
        print(f"\n{'='*70}")
        print(f"ZERO-SHOT: {source_name} → {target_name}")
        print(f"Method: {method_name} | pred_len={cfg.pred_len}")
        print(f"{'='*70}")
    
    def apply_reconstruction(self, data: np.ndarray, dataset_name: str) -> np.ndarray:
        if self.cfg.use_stl:
            print(f"  Applying STL reconstruction (period={STLDecomposition.get_period(dataset_name)})")
            return STLDecomposition.apply_to_multivariate(data, dataset_name, robust=True)
        elif self.cfg.use_wavelet:
            print(f"  Applying Wavelet reconstruction")
            return WaveletReconstruction.apply_to_multivariate(data, "db4", 2, 0.1)
        elif self.cfg.use_ssa:
            print(f"  Applying SSA reconstruction")
            return SSARecovery.apply_to_multivariate(data, 30, 5)
        else:
            print(f"  Baseline: No reconstruction")
            return data.astype(np.float32)
    
    def prepare_data(self):
        print(f"\n[1/4] Loading data...")
        
        source_values = DataManager.load_raw(self.source_name)
        target_values = DataManager.load_raw(self.target_name)
        
        print(f"  Source {self.source_name}: {source_values.shape}")
        print(f"  Target {self.target_name}: {target_values.shape}")
        
        print(f"\n[2/4] Applying reconstruction...")
        source_reconstructed = self.apply_reconstruction(source_values, self.source_name)
        target_reconstructed = self.apply_reconstruction(target_values, self.target_name)
        
        self.cfg.enc_in = source_reconstructed.shape[1]
        print(f"  Channels: source={self.cfg.enc_in}, target={target_reconstructed.shape[1]}")
        
        n_total = len(source_reconstructed)
        n_train = int(n_total * 0.70)
        n_val = int(n_total * 0.10)
        
        train_data = source_reconstructed[:n_train]
        val_data = source_reconstructed[n_train:n_train + n_val]
        
        print(f"\n[3/4] Standardizing using source statistics...")
        self.scaler = StandardScaler()
        self.scaler.fit(train_data)
        
        train_scaled = self.scaler.transform(train_data).astype(np.float32)
        val_scaled = self.scaler.transform(val_data).astype(np.float32)
        
        target_scaled = self.scaler.transform(target_reconstructed).astype(np.float32)
        
        n_test_start = int(len(target_scaled) * 0.70)
        test_data = target_scaled[n_test_start:]
        
        print(f"  Train samples: {len(train_scaled)}")
        print(f"  Val samples: {len(val_scaled)}")
        print(f"  Zero-shot test samples: {len(test_data)}")
        
        self.train_ds = WindowDataset(train_scaled, self.cfg.lookback, self.cfg.pred_len)
        self.val_ds = WindowDataset(val_scaled, self.cfg.lookback, self.cfg.pred_len)
        self.test_ds = WindowDataset(test_data, self.cfg.lookback, self.cfg.pred_len)
        
        self.train_loader = DataLoader(self.train_ds, batch_size=self.cfg.batch_size, shuffle=True)
        self.val_loader = DataLoader(self.val_ds, batch_size=self.cfg.batch_size_test, shuffle=False)
        self.test_loader = DataLoader(self.test_ds, batch_size=self.cfg.batch_size_test, shuffle=False)
        
        print(f"  Train windows: {len(self.train_ds)}")
        print(f"  Val windows: {len(self.val_ds)}")
        print(f"  Test windows: {len(self.test_ds)}")
    
    def train(self):
        print(f"\n[4/4] Training on {self.source_name}...")
        
        self.model = RepoICTSPModel(self.cfg).to(self.device)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"  Model parameters: {total_params:,}")
        
        optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )
        
        best_val_mse = float("inf")
        best_state = None
        patience_counter = 0
        
        for epoch in range(1, self.cfg.train_epochs + 1):
            self.model.train()
            epoch_loss = 0.0
            n_batches = 0
            
            for x_enc, y, x_mark_dec in self.train_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                x_mark_dec = x_mark_dec.to(self.device)
                
                optimizer.zero_grad()
                pred = self.model(x_enc, x_mark_dec)
                loss = F.mse_loss(pred, y)
                loss.backward()
                
                if self.cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                
                optimizer.step()
                epoch_loss += loss.item()
                n_batches += 1
            
            val_metrics = evaluate(self.model, self.val_loader, self.device)
            
            if val_metrics["mse"] < best_val_mse - 1e-8:
                best_val_mse = val_metrics["mse"]
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                patience_counter = 0
                if epoch % 10 == 0:
                    print(f"  Epoch {epoch:3d} | Loss {epoch_loss/n_batches:.6f} | Val MSE {val_metrics['mse']:.6f} ✓")
            else:
                patience_counter += 1
                if epoch % 10 == 0:
                    print(f"  Epoch {epoch:3d} | Loss {epoch_loss/n_batches:.6f} | Val MSE {val_metrics['mse']:.6f}")
            
            if patience_counter >= self.cfg.patience_steps:
                print(f"  Early stopping at epoch {epoch}")
                break
        
        if best_state is not None:
            self.model.load_state_dict(best_state)
        
        print(f"  Best Validation MSE: {best_val_mse:.6f}")
        return best_val_mse
    
    def evaluate_zero_shot(self):
        print(f"\n  Zero-shot evaluation on {self.target_name}...")
        test_metrics = evaluate(self.model, self.test_loader, self.device)
        print(f"  Zero-Shot MSE: {test_metrics['mse']:.6f}")
        print(f"  Zero-Shot MAE: {test_metrics['mae']:.6f}")
        return test_metrics
    
    def evaluate_baselines(self):
        print(f"\n  Baseline evaluation on {self.target_name}...")
        results = {}
        
        lv = LastValueBaseline()
        lv_mse = []
        for x_enc, y, _ in self.test_loader:
            x_enc = x_enc.to(self.device)
            y = y.to(self.device)
            pred = lv(x_enc, self.cfg.pred_len)
            mse, _ = compute_metrics(pred, y)
            lv_mse.append(mse)
        results["LastValue"] = float(np.mean(lv_mse))
        
        nlinear = NLinearBaseline(self.cfg.lookback, self.cfg.pred_len, self.cfg.enc_in).to(self.device)
        opt = torch.optim.Adam(nlinear.parameters(), lr=1e-3)
        
        for _ in range(30):
            nlinear.train()
            for x_enc, y, _ in self.train_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                opt.zero_grad()
                loss = F.mse_loss(nlinear(x_enc), y)
                loss.backward()
                opt.step()
        
        nlinear.eval()
        nlin_mse = []
        with torch.no_grad():
            for x_enc, y, _ in self.test_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                pred = nlinear(x_enc)
                mse, _ = compute_metrics(pred, y)
                nlin_mse.append(mse)
        results["NLinear"] = float(np.mean(nlin_mse))
        
        print(f"  LastValue MSE: {results['LastValue']:.6f}")
        print(f"  NLinear MSE: {results['NLinear']:.6f}")
        
        return results
    
    def plot_prediction(self, save_path=None):
        if len(self.test_loader) == 0:
            return
        
        self.model.eval()
        x_enc, y, x_mark_dec = next(iter(self.test_loader))
        x_enc = x_enc.to(self.device)
        y = y.to(self.device)
        x_mark_dec = x_mark_dec.to(self.device)
        
        with torch.no_grad():
            pred = self.model(x_enc, x_mark_dec)
        
        true = y[0, :, 0].cpu().numpy()
        pred0 = pred[0, :, 0].cpu().numpy()
        
        plt.figure(figsize=(12, 5))
        plt.plot(true, label="True", linewidth=2)
        plt.plot(pred0, label="Zero-Shot Prediction", linewidth=2, linestyle="--")
        plt.title(f"Zero-Shot: {self.source_name} → {self.target_name} | {self.method_name}", fontsize=14)
        plt.xlabel("Horizon Step")
        plt.ylabel("Normalized Value")
        plt.legend()
        plt.grid(True, alpha=0.3)
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
    
    def cleanup(self):
        if hasattr(self, "model"):
            del self.model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================================
# RUN ETTh1 ↔ ETTm1 EXPERIMENTS
# ============================================================================

def run_etth1_ettm1_experiments():
    """Run zero-shot experiments between ETTh1 and ETTm1"""
    
    # Zero-shot directions (source -> target)
    zero_shot_directions = [
        ("ETTh1", "ETTm1"),  # Train on hourly, test on 15-min
        ("ETTm1", "ETTh1"),  # Train on 15-min, test on hourly
    ]
    
    # Methods
    methods = [
        ("Baseline", False, False, False),
        ("STL", True, False, False),
        ("Wavelet", False, True, False),
        ("SSA", False, False, True),
    ]
    
    pred_lens = [96, 192]
    all_results = {}
    
    print("\n" + "🔥" * 50)
    print("ICTSP ZERO-SHOT LEARNING - ETTh1 ↔ ETTm1")
    print(f"Total: 2 directions × 2 pred_lens × 4 methods = 16 experiments")
    print("Methods: Baseline, STL, Wavelet, SSA")
    print("🔥" * 50)
    
    total_exp = len(zero_shot_directions) * len(pred_lens) * len(methods)
    exp_count = 0
    
    for source, target in zero_shot_directions:
        for pred_len in pred_lens:
            for method_name, use_stl, use_wavelet, use_ssa in methods:
                exp_count += 1
                key = f"{source}_to_{target}_{method_name}_pred{pred_len}"
                
                print("\n" + "=" * 70)
                print(f"[{exp_count}/{total_exp}] {source} → {target} | Method: {method_name} | pred_len={pred_len}")
                print("=" * 70)
                
                try:
                    # Create config
                    cfg = Config(
                        dataset_name=source,
                        pred_len=pred_len,
                        use_stl=use_stl,
                        use_wavelet=use_wavelet,
                        use_ssa=use_ssa,
                    )
                    
                    # Create trainer
                    trainer = ZeroShotTrainer(cfg, source, target, method_name)
                    
                    # Prepare data (with reconstruction)
                    trainer.prepare_data()
                    
                    # Train on source
                    best_val_mse = trainer.train()
                    
                    # Evaluate zero-shot on target
                    test_metrics = trainer.evaluate_zero_shot()
                    
                    # Evaluate baselines
                    baseline_metrics = trainer.evaluate_baselines()
                    
                    # Plot
                    trainer.plot_prediction(save_path=f"zero_shot_{source}_to_{target}_{method_name}_pred{pred_len}.png")
                    
                    # Store results
                    all_results[key] = {
                        "source": source,
                        "target": target,
                        "method": method_name,
                        "pred_len": pred_len,
                        "source_channels": cfg.enc_in,
                        "ictsp_zero_shot_mse": test_metrics["mse"],
                        "ictsp_zero_shot_mae": test_metrics["mae"],
                        "best_val_mse": best_val_mse,
                        "lastvalue_mse": baseline_metrics["LastValue"],
                        "nlinear_mse": baseline_metrics["NLinear"],
                    }
                    
                    trainer.cleanup()
                    
                    # Save progress after each experiment
                    with open("etth1_ettm1_progress.json", "w") as f:
                        json.dump(all_results, f, indent=2, default=str)
                    
                except Exception as e:
                    print(f"❌ Error in {key}: {e}")
                    import traceback
                    traceback.print_exc()
                    all_results[key] = {"error": str(e)}
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
    
    return all_results


# ============================================================================
# RESULTS VISUALIZATION
# ============================================================================

def print_results(results):
    print("\n" + "=" * 100)
    print("ETTh1 ↔ ETTm1 ZERO-SHOT LEARNING RESULTS")
    print("=" * 100)
    
    results_list = []
    for key, val in results.items():
        if isinstance(val, dict) and "ictsp_zero_shot_mse" in val:
            results_list.append(val)
    
    if results_list:
        df = pd.DataFrame(results_list)
        
        # Results for pred_len=96
        print("\n" + "=" * 80)
        print("PREDICTION LENGTH: 96")
        print("=" * 80)
        
        df_96 = df[df['pred_len'] == 96]
        if len(df_96) > 0:
            pivot_96 = df_96.pivot_table(
                values='ictsp_zero_shot_mse',
                index=['source', 'target'],
                columns='method',
                aggfunc='first'
            )
            print(pivot_96.round(6))
        
        # Results for pred_len=192
        print("\n" + "=" * 80)
        print("PREDICTION LENGTH: 192")
        print("=" * 80)
        
        df_192 = df[df['pred_len'] == 192]
        if len(df_192) > 0:
            pivot_192 = df_192.pivot_table(
                values='ictsp_zero_shot_mse',
                index=['source', 'target'],
                columns='method',
                aggfunc='first'
            )
            print(pivot_192.round(6))
        
        # Save to CSV
        df.to_csv("etth1_ettm1_results.csv", index=False)
        print("\n✅ Saved results to etth1_ettm1_results.csv")
        
        # Create summary plot
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        directions = [("ETTh1", "ETTm1"), ("ETTm1", "ETTh1")]
        pred_lens = [96, 192]
        
        for i, (source, target) in enumerate(directions):
            for j, pred_len in enumerate(pred_lens):
                ax = axes[i, j]
                sub_df = df[(df['source'] == source) & (df['target'] == target) & (df['pred_len'] == pred_len)]
                
                if len(sub_df) > 0:
                    methods_list = ['Baseline', 'STL', 'Wavelet', 'SSA']
                    mse_values = []
                    
                    for method in methods_list:
                        val = sub_df[sub_df['method'] == method]['ictsp_zero_shot_mse'].values
                        mse_values.append(val[0] if len(val) > 0 else 0)
                    
                    bars = ax.bar(methods_list, mse_values, 
                                 color=['steelblue', 'forestgreen', 'darkorange', 'crimson'])
                    ax.set_title(f'{source} → {target} | pred_len={pred_len}', fontsize=12)
                    ax.set_ylabel('MSE')
                    ax.grid(True, alpha=0.3)
                    
                    for bar, val in zip(bars, mse_values):
                        if val > 0:
                            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                                   f'{val:.4f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig("etth1_ettm1_results_summary.png", dpi=150, bbox_inches="tight")
        plt.show()
        
    else:
        print("No valid results to display")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("\n" + "🔄" * 50)
    print("ICTSP ZERO-SHOT LEARNING - ETTh1 ↔ ETTm1")
    print("=" * 60)
    print("Zero-Shot Directions (2):")
    print("  1. ETTh1 → ETTm1 (Train on hourly, test on 15-min)")
    print("  2. ETTm1 → ETTh1 (Train on 15-min, test on hourly)")
    print("-" * 60)
    print("Methods (4): Baseline, STL, Wavelet, SSA")
    print("Prediction Lengths: 96, 192")
    print(f"Total Experiments: 2 directions × 2 pred_lens × 4 methods = 16")
    print("🔄" * 50)
    
    # Run experiments
    results = run_etth1_ettm1_experiments()
    
    # Print and save results
    print_results(results)
    
    # Save full results
    with open("etth1_ettm1_all_results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    
    print("\n✅ All ETTh1 ↔ ETTm1 zero-shot experiments completed!")
    print("   - etth1_ettm1_results.csv (tabular results)")
    print("   - etth1_ettm1_all_results.json (full results)")
    print("   - etth1_ettm1_results_summary.png (visualization)")

PyTorch version: 2.10.0+cu128
CUDA available: True
Using device: cpu (CPU for stability)

🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄
ICTSP ZERO-SHOT LEARNING - ETTh1 ↔ ETTm1
Zero-Shot Directions (2):
  1. ETTh1 → ETTm1 (Train on hourly, test on 15-min)
  2. ETTm1 → ETTh1 (Train on 15-min, test on hourly)
------------------------------------------------------------
Methods (4): Baseline, STL, Wavelet, SSA
Prediction Lengths: 96, 192
Total Experiments: 2 directions × 2 pred_lens × 4 methods = 16
🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
ICTSP ZERO-SHOT LEARNING - ETTh1 ↔ ETTm1
Total: 2 directions × 2 pred_lens × 4 methods = 16 experiments
Methods: Baseline, STL, Wavelet, SSA
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

[1/16] ETTh1 → ETTm1 | Method: Baseline | pred_len=96

ZERO-SHOT: ETTh1 → ETTm1
Method: Baseline | pred_len=96

[1/4] Loading data...
Loading ETTh1 from https://raw.githubusercontent.com/zhouhaoyi/

In [1]:
"""
ICTSP ZERO-SHOT LEARNING - Run ONLY the 16th (missing) experiment
Experiment 16: ETTh1 → ETTm1 | Method: SSA | pred_len=192
"""

import os
import gc
import json
import time
import random
import warnings
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.interpolate import interp1d

warnings.filterwarnings("ignore")

# ============================================================================
# DEVICE SETUP
# ============================================================================
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = "cpu"
print(f"Using device: {device} (CPU for stability)")

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(2024)

# ============================================================================
# OPTIONAL PACKAGES
# ============================================================================
try:
    import pywt
    HAS_PYWT = True
except ImportError:
    HAS_PYWT = False
    print("PyWavelets not found. Installing...")
    subprocess.check_call(["pip", "install", "-q", "PyWavelets"])
    import pywt
    HAS_PYWT = True

try:
    from statsmodels.tsa.seasonal import STL as STLModel
    HAS_STL = True
except ImportError:
    HAS_STL = False
    print("statsmodels not found. Installing...")
    subprocess.check_call(["pip", "install", "-q", "statsmodels"])
    from statsmodels.tsa.seasonal import STL as STLModel
    HAS_STL = True


# ============================================================================
# RECONSTRUCTION METHODS
# ============================================================================

class STLDecomposition:
    @staticmethod
    def get_period(dataset_name: str) -> int:
        if "ETTh" in dataset_name:
            return 24  # Hourly data: 24 hours/day
        elif "ETTm" in dataset_name:
            return 96  # 15-min data: 96 periods/day
        return 24

    @staticmethod
    def denoise_signal(signal: np.ndarray, period: int, robust: bool = True) -> np.ndarray:
        if not HAS_STL:
            return signal
        signal = signal.copy()
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8 or len(signal) < 2 * period:
            return signal.astype(np.float32)
        try:
            stl = STLModel(signal, period=period, robust=robust)
            result = stl.fit()
            denoised = result.trend + result.seasonal
            bad = np.isnan(denoised)
            denoised[bad] = signal[bad]
            return denoised.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, dataset_name: str, robust: bool = True) -> np.ndarray:
        period = STLDecomposition.get_period(dataset_name)
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = STLDecomposition.denoise_signal(data[:, c], period, robust)
        return out


class WaveletReconstruction:
    @staticmethod
    def denoise_signal(signal: np.ndarray, wavelet: str = "db4", level: int = 2, threshold_factor: float = 0.1) -> np.ndarray:
        if not HAS_PYWT:
            return signal
        signal = signal.copy()
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8:
            return signal.astype(np.float32)
        max_level = max(1, int(np.log2(len(signal))) - 1)
        level = min(level, max_level)
        try:
            coeffs = pywt.wavedec(signal, wavelet, level=level)
            if len(coeffs) > 1:
                sigma = np.median(np.abs(coeffs[-1])) / 0.6745
                threshold = sigma * np.sqrt(2 * np.log(len(signal))) * threshold_factor
                coeffs_thresh = [coeffs[0]]
                for i in range(1, len(coeffs)):
                    coeffs_thresh.append(pywt.threshold(coeffs[i], threshold, mode="soft"))
                out = pywt.waverec(coeffs_thresh, wavelet)[:len(signal)]
            else:
                out = signal
            return out.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, wavelet: str = "db4", level: int = 2, threshold_factor: float = 0.1) -> np.ndarray:
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = WaveletReconstruction.denoise_signal(data[:, c], wavelet, level, threshold_factor)
        return out


class SSARecovery:
    @staticmethod
    def _embed(signal: np.ndarray, L: int) -> np.ndarray:
        N = len(signal)
        K = N - L + 1
        X = np.zeros((L, K), dtype=np.float32)
        for i in range(L):
            X[i, :] = signal[i:i + K]
        return X

    @staticmethod
    def _diagonal_averaging(X: np.ndarray) -> np.ndarray:
        L, K = X.shape
        N = L + K - 1
        signal = np.zeros(N, dtype=np.float32)
        counts = np.zeros(N, dtype=np.float32)
        for i in range(L):
            for j in range(K):
                signal[i + j] += X[i, j]
                counts[i + j] += 1.0
        return signal / np.maximum(counts, 1e-8)

    @staticmethod
    def denoise_signal(signal: np.ndarray, L: int = 30, r: int = 5) -> np.ndarray:
        signal = signal.copy()
        N = len(signal)
        if N < L:
            L = max(2, N // 2)
        if np.any(np.isnan(signal)):
            mask = np.isnan(signal)
            if np.any(~mask):
                signal[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), signal[~mask])
            else:
                return signal.astype(np.float32)
        if np.std(signal) < 1e-8:
            return signal.astype(np.float32)
        try:
            X = SSARecovery._embed(signal, L)
            U, S, Vt = np.linalg.svd(X, full_matrices=False)
            r = min(r, len(S))
            Xr = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]
            out = SSARecovery._diagonal_averaging(Xr)[:N]
            return out.astype(np.float32)
        except Exception:
            return signal.astype(np.float32)

    @staticmethod
    def apply_to_multivariate(data: np.ndarray, L: int = 30, r: int = 5) -> np.ndarray:
        out = np.zeros_like(data, dtype=np.float32)
        for c in range(data.shape[1]):
            out[:, c] = SSARecovery.denoise_signal(data[:, c], L, r)
        return out


# ============================================================================
# DATA LOADING
# ============================================================================

class DataManager:
    DATASET_URLS = {
        "ETTh1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv",
        "ETTm1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv",
    }
    
    @staticmethod
    def load_raw(name: str) -> np.ndarray:
        url = DataManager.DATASET_URLS[name]
        print(f"Loading {name} from {url}...")
        df = pd.read_csv(url)
        if "date" in df.columns:
            df = df.drop(columns=["date"])
        values = df.values.astype(np.float32)
        print(f"✅ Loaded {name}: {values.shape[1]} channels, {values.shape[0]} time steps")
        return values


class WindowDataset(Dataset):
    def __init__(self, data: np.ndarray, seq_len: int, pred_len: int):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.window_len = seq_len + pred_len
        
        if len(data) >= self.window_len:
            self.n_windows = len(data) - self.window_len + 1
        else:
            self.n_windows = 0
    
    def __len__(self):
        return self.n_windows
    
    def __getitem__(self, idx):
        window = self.data[idx:idx + self.window_len]
        x_enc = window[:self.seq_len]
        y = window[self.seq_len:self.seq_len + self.pred_len]
        x_mark_dec_dummy = torch.zeros(self.seq_len + self.pred_len, 1)
        return x_enc, y, x_mark_dec_dummy


# ============================================================================
# ICTSP CORE
# ============================================================================

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_size, heads, dropout=0.1):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        self.dropout = nn.Dropout(dropout)
        assert self.head_dim * heads == embed_size
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)
    
    def forward(self, values, keys, queries, mask=None):
        N = queries.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], queries.shape[1]
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = queries.reshape(N, query_len, self.heads, self.head_dim)
        values = self.values(values)
        keys = self.dropout(self.keys(keys))
        queries = self.queries(queries)
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))
        energy = energy / (self.embed_size ** 0.5)
        attention = F.softmax(energy, dim=-1)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )
        out = self.fc_out(out)
        return out, attention


class TransformerEncoder(nn.Module):
    def __init__(self, emb_size=128, depth=3, heads=8, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=emb_size,
                nhead=heads,
                dim_feedforward=mlp_ratio * emb_size,
                batch_first=True,
                dropout=dropout,
                norm_first=False,
                activation="gelu",
            )
            for _ in range(depth)
        ])
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class Tokenizer(nn.Module):
    def __init__(self, lookback=512, output=96, stride=8):
        super().__init__()
        self.d = lookback + output
        self.s = stride
    
    def forward(self, tensor):
        B, C, L = tensor.shape
        if L >= self.d:
            unfolded = tensor.flip(-1).unfold(dimension=2, size=self.d, step=self.s)
            return unfolded.flip(-1).flip(-2)
        else:
            padding = torch.zeros(B, C, self.d - L, device=tensor.device, dtype=tensor.dtype)
            tensor_padded = torch.cat([tensor, padding], dim=-1)
            unfolded = tensor_padded.flip(-1).unfold(dimension=2, size=self.d, step=self.s)
            return unfolded.flip(-1).flip(-2)


class ICTSP(nn.Module):
    def __init__(
        self,
        lookback=512,
        output=96,
        depth=3,
        heads=8,
        mlp_ratio=4,
        d_model=128,
        external_stride=8,
        n_channels=7,
        dropout=0.5,
    ):
        super().__init__()
        self.lookback = lookback
        self.pred_len = output
        self.external_stride = external_stride
        self.input_projection = nn.Linear(lookback + output, d_model)
        self.transformer_encoder = TransformerEncoder(d_model, depth, heads, mlp_ratio, dropout)
        self.input_norm = nn.LayerNorm(d_model)
        self.output_norm = nn.LayerNorm(d_model)
        self.output_embedding = nn.Parameter(0.01 * torch.randn(1, 1, 1200))
        self.output_projection = nn.Linear(d_model, output)
        self.channel_embedding = nn.Parameter(0.01 * torch.randn(1, 1024, d_model))
        self.pos_embedding = nn.Parameter(0.01 * torch.randn(1, 8192, 1, d_model))
        self.pos_embedding_after = nn.Parameter(0.01 * torch.randn(1, 8192, d_model))
        self.linear_warmup_steps = 5000
        self.linear_warm_up_counter = 0
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
    
    def forward(self, x, x_mark_dec=None):
        lookback = self.lookback
        future = self.pred_len
        mean = x[:, [-1], :].detach()
        x = x.permute(0, 2, 1)
        output_embedding = self.output_embedding[:, :, 0:future].expand(x.shape[0], x.shape[1], -1)
        x = torch.cat([x, output_embedding + mean.permute(0, 2, 1)], dim=-1)
        B, C, _ = x.shape
        number_of_targets = C
        x_orig = x[:, :, 0:-future].clone()
        external_tokenizer = Tokenizer(lookback, future, stride=self.external_stride)
        ex_tokens = external_tokenizer(x_orig)
        _, _, _, d = ex_tokens.shape
        ex_tokens = ex_tokens.permute(0, 2, 1, 3).reshape(B, -1, d)
        x_target = x[:, -number_of_targets:, -(lookback + future):]
        x_tokens = torch.cat([ex_tokens, x_target], dim=1)
        token_mean = x_tokens[:, :, [-(future + 1)]].detach()
        x_tokens = x_tokens - token_mean
        x_tokens = self.input_projection(x_tokens)
        channel_mask = self.channel_embedding[:, -C:, :]
        x_tokens = x_tokens + channel_mask.repeat(1, x_tokens.shape[1] // C, 1)
        pos_emb = self.pos_embedding[:, -(x_tokens.shape[1] // C):, :, :].expand(-1, -1, C, -1)
        pos_emb = pos_emb.reshape(pos_emb.shape[0], pos_emb.shape[1] * pos_emb.shape[2], pos_emb.shape[3])
        x_tokens = x_tokens + pos_emb
        
        if self.linear_warm_up_counter < self.linear_warmup_steps:
            if self.training:
                self.linear_warm_up_counter += 1
            x_output = self.output_projection(x_tokens[:, -number_of_targets:, :])
            x_output = x_output + token_mean[:, -x_output.shape[1]:, :]
            x_output = x_output.permute(0, 2, 1)
            return x_output
        
        x_tokens = self.input_norm(x_tokens)
        x_tokens = self.transformer_encoder(x_tokens)
        x_output = x_tokens[:, -number_of_targets:, :]
        x_output = self.output_norm(x_output)
        x_output = self.output_projection(x_output)
        x_output = x_output + token_mean[:, -x_output.shape[1]:, :]
        x_output = x_output.permute(0, 2, 1)
        return x_output


class RepoICTSPModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = ICTSP(
            lookback=cfg.lookback,
            output=cfg.pred_len,
            depth=cfg.e_layers,
            heads=cfg.n_heads,
            mlp_ratio=cfg.mlp_ratio,
            d_model=cfg.d_model,
            external_stride=cfg.sampling_step,
            n_channels=cfg.enc_in,
            dropout=cfg.dropout,
        )
    
    def forward(self, x, x_mark_dec):
        return self.model(x, x_mark_dec)


# ============================================================================
# BASELINE MODELS (for comparison)
# ============================================================================

class LastValueBaseline(nn.Module):
    def forward(self, history, horizon):
        return history[:, -1:, :].repeat(1, horizon, 1)


class NLinearBaseline(nn.Module):
    def __init__(self, seq_len, horizon, num_channels):
        super().__init__()
        self.linears = nn.ModuleList([nn.Linear(seq_len, horizon) for _ in range(num_channels)])
    
    def forward(self, history):
        return torch.stack([self.linears[c](history[:, :, c]) for c in range(history.shape[-1])], dim=2)


# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    dataset_name: str = "ETTh1"
    experiment_mode: str = "zero_shot"
    use_stl: bool = False
    use_wavelet: bool = False
    use_ssa: bool = False
    lookback: int = 512
    pred_len: int = 96
    e_layers: int = 3
    d_model: int = 128
    n_heads: int = 8
    mlp_ratio: int = 4
    dropout: float = 0.5
    sampling_step: int = 8
    batch_size: int = 32
    batch_size_test: int = 32
    learning_rate: float = 5e-4
    weight_decay: float = 1e-5
    train_epochs: int = 100
    patience_steps: int = 20
    grad_clip: float = 1.0
    enc_in: int = 7
    device: str = device
    seed: int = 2024

    
    def __post_init__(self):
        self.lookback = 512
        self.batch_size = 32
        self.train_epochs = 100
        self.patience_steps = 20


# ============================================================================
# METRICS & EVALUATION
# ============================================================================

def compute_metrics(pred, target):
    with torch.no_grad():
        mse = F.mse_loss(pred, target).item()
        mae = F.l1_loss(pred, target).item()
    return mse, mae


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_mse = 0.0
    total_mae = 0.0
    n_batches = 0
    
    for x_enc, y, x_mark_dec in loader:
        x_enc = x_enc.to(device)
        y = y.to(device)
        x_mark_dec = x_mark_dec.to(device)
        pred = model(x_enc, x_mark_dec)
        mse, mae = compute_metrics(pred, y)
        total_mse += mse
        total_mae += mae
        n_batches += 1
    
    return {
        "mse": total_mse / n_batches if n_batches > 0 else float("inf"),
        "mae": total_mae / n_batches if n_batches > 0 else float("inf"),
    }


# ============================================================================
# ZERO-SHOT TRAINER
# ============================================================================

class ZeroShotTrainer:
    def __init__(self, cfg: Config, source_name: str, target_name: str, method_name: str):
        self.cfg = cfg
        self.device = cfg.device
        self.source_name = source_name
        self.target_name = target_name
        self.method_name = method_name
        set_seed(cfg.seed)
        
        print(f"\n{'='*70}")
        print(f"ZERO-SHOT: {source_name} → {target_name}")
        print(f"Method: {method_name} | pred_len={cfg.pred_len}")
        print(f"{'='*70}")
    
    def apply_reconstruction(self, data: np.ndarray, dataset_name: str) -> np.ndarray:
        if self.cfg.use_stl:
            print(f"  Applying STL reconstruction (period={STLDecomposition.get_period(dataset_name)})")
            return STLDecomposition.apply_to_multivariate(data, dataset_name, robust=True)
        elif self.cfg.use_wavelet:
            print(f"  Applying Wavelet reconstruction")
            return WaveletReconstruction.apply_to_multivariate(data, "db4", 2, 0.1)
        elif self.cfg.use_ssa:
            print(f"  Applying SSA reconstruction")
            return SSARecovery.apply_to_multivariate(data, 30, 5)
        else:
            print(f"  Baseline: No reconstruction")
            return data.astype(np.float32)
    
    def prepare_data(self):
        print(f"\n[1/4] Loading data...")
        
        source_values = DataManager.load_raw(self.source_name)
        target_values = DataManager.load_raw(self.target_name)
        
        print(f"  Source {self.source_name}: {source_values.shape}")
        print(f"  Target {self.target_name}: {target_values.shape}")
        
        print(f"\n[2/4] Applying reconstruction...")
        source_reconstructed = self.apply_reconstruction(source_values, self.source_name)
        target_reconstructed = self.apply_reconstruction(target_values, self.target_name)
        
        self.cfg.enc_in = source_reconstructed.shape[1]
        print(f"  Channels: source={self.cfg.enc_in}, target={target_reconstructed.shape[1]}")
        
        n_total = len(source_reconstructed)
        n_train = int(n_total * 0.70)
        n_val = int(n_total * 0.10)
        
        train_data = source_reconstructed[:n_train]
        val_data = source_reconstructed[n_train:n_train + n_val]
        
        print(f"\n[3/4] Standardizing using source statistics...")
        self.scaler = StandardScaler()
        self.scaler.fit(train_data)
        
        train_scaled = self.scaler.transform(train_data).astype(np.float32)
        val_scaled = self.scaler.transform(val_data).astype(np.float32)
        
        target_scaled = self.scaler.transform(target_reconstructed).astype(np.float32)
        
        n_test_start = int(len(target_scaled) * 0.70)
        test_data = target_scaled[n_test_start:]
        
        print(f"  Train samples: {len(train_scaled)}")
        print(f"  Val samples: {len(val_scaled)}")
        print(f"  Zero-shot test samples: {len(test_data)}")
        
        self.train_ds = WindowDataset(train_scaled, self.cfg.lookback, self.cfg.pred_len)
        self.val_ds = WindowDataset(val_scaled, self.cfg.lookback, self.cfg.pred_len)
        self.test_ds = WindowDataset(test_data, self.cfg.lookback, self.cfg.pred_len)
        
        self.train_loader = DataLoader(self.train_ds, batch_size=self.cfg.batch_size, shuffle=True)
        self.val_loader = DataLoader(self.val_ds, batch_size=self.cfg.batch_size_test, shuffle=False)
        self.test_loader = DataLoader(self.test_ds, batch_size=self.cfg.batch_size_test, shuffle=False)
        
        print(f"  Train windows: {len(self.train_ds)}")
        print(f"  Val windows: {len(self.val_ds)}")
        print(f"  Test windows: {len(self.test_ds)}")
    
    def train(self):
        print(f"\n[4/4] Training on {self.source_name}...")
        
        self.model = RepoICTSPModel(self.cfg).to(self.device)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"  Model parameters: {total_params:,}")
        
        optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=self.cfg.learning_rate,
            weight_decay=self.cfg.weight_decay,
        )
        
        best_val_mse = float("inf")
        best_state = None
        patience_counter = 0
        
        for epoch in range(1, self.cfg.train_epochs + 1):
            self.model.train()
            epoch_loss = 0.0
            n_batches = 0
            
            for x_enc, y, x_mark_dec in self.train_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                x_mark_dec = x_mark_dec.to(self.device)
                
                optimizer.zero_grad()
                pred = self.model(x_enc, x_mark_dec)
                loss = F.mse_loss(pred, y)
                loss.backward()
                
                if self.cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                
                optimizer.step()
                epoch_loss += loss.item()
                n_batches += 1
            
            val_metrics = evaluate(self.model, self.val_loader, self.device)
            
            if val_metrics["mse"] < best_val_mse - 1e-8:
                best_val_mse = val_metrics["mse"]
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                patience_counter = 0
                if epoch % 10 == 0:
                    print(f"  Epoch {epoch:3d} | Loss {epoch_loss/n_batches:.6f} | Val MSE {val_metrics['mse']:.6f} ✓")
            else:
                patience_counter += 1
                if epoch % 10 == 0:
                    print(f"  Epoch {epoch:3d} | Loss {epoch_loss/n_batches:.6f} | Val MSE {val_metrics['mse']:.6f}")
            
            if patience_counter >= self.cfg.patience_steps:
                print(f"  Early stopping at epoch {epoch}")
                break
        
        if best_state is not None:
            self.model.load_state_dict(best_state)
        
        print(f"  Best Validation MSE: {best_val_mse:.6f}")
        return best_val_mse
    
    def evaluate_zero_shot(self):
        print(f"\n  Zero-shot evaluation on {self.target_name}...")
        test_metrics = evaluate(self.model, self.test_loader, self.device)
        print(f"  Zero-Shot MSE: {test_metrics['mse']:.6f}")
        print(f"  Zero-Shot MAE: {test_metrics['mae']:.6f}")
        return test_metrics
    
    def evaluate_baselines(self):
        print(f"\n  Baseline evaluation on {self.target_name}...")
        results = {}
        
        lv = LastValueBaseline()
        lv_mse = []
        for x_enc, y, _ in self.test_loader:
            x_enc = x_enc.to(self.device)
            y = y.to(self.device)
            pred = lv(x_enc, self.cfg.pred_len)
            mse, _ = compute_metrics(pred, y)
            lv_mse.append(mse)
        results["LastValue"] = float(np.mean(lv_mse))
        
        nlinear = NLinearBaseline(self.cfg.lookback, self.cfg.pred_len, self.cfg.enc_in).to(self.device)
        opt = torch.optim.Adam(nlinear.parameters(), lr=1e-3)
        
        for _ in range(30):
            nlinear.train()
            for x_enc, y, _ in self.train_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                opt.zero_grad()
                loss = F.mse_loss(nlinear(x_enc), y)
                loss.backward()
                opt.step()
        
        nlinear.eval()
        nlin_mse = []
        with torch.no_grad():
            for x_enc, y, _ in self.test_loader:
                x_enc = x_enc.to(self.device)
                y = y.to(self.device)
                pred = nlinear(x_enc)
                mse, _ = compute_metrics(pred, y)
                nlin_mse.append(mse)
        results["NLinear"] = float(np.mean(nlin_mse))
        
        print(f"  LastValue MSE: {results['LastValue']:.6f}")
        print(f"  NLinear MSE: {results['NLinear']:.6f}")
        
        return results
    
    def plot_prediction(self, save_path=None):
        if len(self.test_loader) == 0:
            return
        
        self.model.eval()
        x_enc, y, x_mark_dec = next(iter(self.test_loader))
        x_enc = x_enc.to(self.device)
        y = y.to(self.device)
        x_mark_dec = x_mark_dec.to(self.device)
        
        with torch.no_grad():
            pred = self.model(x_enc, x_mark_dec)
        
        true = y[0, :, 0].cpu().numpy()
        pred0 = pred[0, :, 0].cpu().numpy()
        
        plt.figure(figsize=(12, 5))
        plt.plot(true, label="True", linewidth=2)
        plt.plot(pred0, label="Zero-Shot Prediction", linewidth=2, linestyle="--")
        plt.title(f"Zero-Shot: {self.source_name} → {self.target_name} | {self.method_name}", fontsize=14)
        plt.xlabel("Horizon Step")
        plt.ylabel("Normalized Value")
        plt.legend()
        plt.grid(True, alpha=0.3)
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
    
    def cleanup(self):
        if hasattr(self, "model"):
            del self.model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================================
# RUN ONLY THE 16th EXPERIMENT
# ============================================================================

def run_16th_experiment():
    """
    Run ONLY the 16th experiment:
    Direction: ETTh1 → ETTm1
    Method: SSA
    Prediction Length: 192
    """
    
    print("\n" + "🔥" * 50)
    print("RUNNING ONLY THE 16th EXPERIMENT")
    print("=" * 50)
    print("Experiment 16/16 Details:")
    print("  → Direction: ETTh1 → ETTm1")
    print("  → Method: SSA") 
    print("  → Prediction Length: 192")
    print("🔥" * 50)
    
    # Experiment 16 parameters
    source = "ETTh1"
    target = "ETTm1"
    pred_len = 192
    method_name = "SSA"
    use_stl = False
    use_wavelet = False
    use_ssa = True
    
    exp_key = f"{source}_to_{target}_{method_name}_pred{pred_len}"
    
    print(f"\n{'='*70}")
    print(f"[1/1] {source} → {target} | Method: {method_name} | pred_len={pred_len}")
    print(f"      This is the 16th (final) experiment")
    print("="*70)
    
    results = {}
    
    try:
        # Create config
        cfg = Config(
            dataset_name=source,
            pred_len=pred_len,
            use_stl=use_stl,
            use_wavelet=use_wavelet,
            use_ssa=use_ssa,
        )
        
        # Create trainer
        trainer = ZeroShotTrainer(cfg, source, target, method_name)
        
        # Prepare data (with SSA reconstruction)
        trainer.prepare_data()
        
        # Train on source (ETTh1)
        best_val_mse = trainer.train()
        
        # Evaluate zero-shot on target (ETTm1)
        test_metrics = trainer.evaluate_zero_shot()
        
        # Evaluate baselines
        baseline_metrics = trainer.evaluate_baselines()
        
        # Plot
        plot_path = f"zero_shot_16th_{source}_to_{target}_{method_name}_pred{pred_len}.png"
        trainer.plot_prediction(save_path=plot_path)
        
        # Store results
        results[exp_key] = {
            "source": source,
            "target": target,
            "method": method_name,
            "pred_len": pred_len,
            "source_channels": cfg.enc_in,
            "ictsp_zero_shot_mse": test_metrics["mse"],
            "ictsp_zero_shot_mae": test_metrics["mae"],
            "best_val_mse": best_val_mse,
            "lastvalue_mse": baseline_metrics["LastValue"],
            "nlinear_mse": baseline_metrics["NLinear"],
            "experiment_number": 16,
            "total_experiments": 16,
        }
        
        trainer.cleanup()
        
        # Save results
        with open("16th_experiment_results.json", "w") as f:
            json.dump(results, f, indent=2, default=str)
        
        print("\n" + "="*70)
        print("✅ 16th EXPERIMENT COMPLETED SUCCESSFULLY!")
        print("="*70)
        print(f"\n📊 Results Summary for Experiment 16:")
        print(f"  → ICTSP Zero-Shot MSE: {test_metrics['mse']:.6f}")
        print(f"  → ICTSP Zero-Shot MAE: {test_metrics['mae']:.6f}")
        print(f"  → Best Validation MSE: {best_val_mse:.6f}")
        print(f"  → LastValue Baseline MSE: {baseline_metrics['LastValue']:.6f}")
        print(f"  → NLinear Baseline MSE: {baseline_metrics['NLinear']:.6f}")
        print(f"\n💾 Results saved to:")
        print(f"  → 16th_experiment_results.json")
        print(f"  → {plot_path}")
        
    except Exception as e:
        print(f"❌ Error in experiment 16: {e}")
        import traceback
        traceback.print_exc()
        results[exp_key] = {"error": str(e), "experiment_number": 16}
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return results


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("\n" + "🔄" * 50)
    print("ICTSP ZERO-SHOT LEARNING - RUN ONLY 16th EXPERIMENT")
    print("=" * 60)
    print("This script runs ONLY the 16th (final) experiment:")
    print("  • Direction: ETTh1 → ETTm1")
    print("  • Method: SSA")
    print("  • Prediction Length: 192")
    print("=" * 60)
    print("All previous experiments (1-15) are SKIPPED.")
    print("🔄" * 50)
    
    # Run only the 16th experiment
    results = run_16th_experiment()
    
    # Display final summary
    print("\n" + "="*70)
    print("FINAL SUMMARY - 16th EXPERIMENT COMPLETE")
    print("="*70)
    
    for key, val in results.items():
        if isinstance(val, dict) and "ictsp_zero_shot_mse" in val:
            print(f"\n✅ Experiment 16/16: {key}")
            print(f"   → MSE: {val['ictsp_zero_shot_mse']:.6f}")
            print(f"   → MAE: {val['ictsp_zero_shot_mae']:.6f}")
            print(f"   → Outperformed LastValue by: {(val['lastvalue_mse'] - val['ictsp_zero_shot_mse'])/val['lastvalue_mse']*100:.1f}%")
    
    print("\n" + "="*70)
    print("🎉 All 16 experiments can now be combined!")
    print("   The 16th experiment results are ready in '16th_experiment_results.json'")
    print("="*70)

PyTorch version: 2.10.0+cu128
CUDA available: True
Using device: cpu (CPU for stability)

🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄
ICTSP ZERO-SHOT LEARNING - RUN ONLY 16th EXPERIMENT
This script runs ONLY the 16th (final) experiment:
  • Direction: ETTh1 → ETTm1
  • Method: SSA
  • Prediction Length: 192
All previous experiments (1-15) are SKIPPED.
🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
RUNNING ONLY THE 16th EXPERIMENT
Experiment 16/16 Details:
  → Direction: ETTh1 → ETTm1
  → Method: SSA
  → Prediction Length: 192
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

[1/1] ETTh1 → ETTm1 | Method: SSA | pred_len=192
      This is the 16th (final) experiment

ZERO-SHOT: ETTh1 → ETTm1
Method: SSA | pred_len=192

[1/4] Loading data...
Loading ETTh1 from https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv...
✅ Loaded ETTh1: 7 channels, 17420 time steps
Loading ETTm1 from https://raw.githubuse